# Level Zero baseline: Muon + Auxiliary AdamW

This notebook runs the three matched seeds for **`muon`**, analyzes the
optimizer in isolation, and publishes its core CSV artifacts to the shared
`baseline_reference` directory.

Trajectory shading is a **Bollinger-style across-seed envelope**: mean ± 2
sample standard deviations. It is not a rolling smoother. Test loss,
perplexity, and exact next-token accuracy are measured at the preregistered
integer-epoch checkpoints and are never used for optimizer updates or
checkpoint selection.

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display

cwd = Path.cwd().resolve()
if (cwd / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd
elif (cwd.parent / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd.parent
elif (cwd / "level_0_baseline" / "configs" / "level0.yaml").is_file():
    EXPERIMENT_ROOT = cwd / "level_0_baseline"
else:
    raise FileNotFoundError(
        "Run from the repository or level_0_baseline tree"
    )

sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))

from level0_baseline.analysis import (
    load_metrics,
    load_spectral_metrics,
    load_test_results,
    plot_final_test_ci,
    plot_optimizer_band,
    run_directory,
    run_status_table,
    test_summary_table,
)
from level0_baseline.baseline_store import (
    default_baseline_store_root,
    export_optimizer_core_results,
    load_epoch_metrics,
)
from level0_baseline.config import canonical_seeds, load_config
from level0_baseline.generate import (
    generate_from_checkpoint,
    write_samples,
)
from level0_baseline.runner import run_suite

CONFIG_PATH = EXPERIMENT_ROOT / "configs" / "level0.yaml"
CONFIG = load_config(CONFIG_PATH)
SEEDS = canonical_seeds(CONFIG)
ROOT = Path(
    os.environ.get(
        "NANOGPT_LEVEL0_ROOT",
        "/tmp/nanogpt-level0-baselines",
    )
)
DATA_ROOT = Path(
    os.environ.get("NANOGPT_LEVEL0_DATA_ROOT", ROOT / "data")
)
RESULTS_ROOT = Path(
    os.environ.get("NANOGPT_LEVEL0_RESULTS_ROOT", ROOT / "results")
)
BASELINE_STORE = default_baseline_store_root()
DEVICE = os.environ.get("NANOGPT_LEVEL0_DEVICE", "mps")
BAND_SIGMA = float(CONFIG["analysis"]["bollinger_sigma"])

print(f"experiment root: {EXPERIMENT_ROOT}")
print(f"data root:       {DATA_ROOT}")
print(f"results root:    {RESULTS_ROOT}")
print(f"baseline store:  {BASELINE_STORE}")
print(f"device:          {DEVICE}")

In [ ]:
OPTIMIZER = "muon"
display(
    run_status_table(
        RESULTS_ROOT,
        optimizers=[OPTIMIZER],
        seeds=SEEDS,
    )
)

## Run or resume the three seeds

Runs are sequential to avoid simultaneous MPS pressure. Completed runs are
skipped; incomplete runs resume from `checkpoint_latest.pt`.

In [ ]:
RUN_TRAINING = True

if RUN_TRAINING:
    display(
        run_suite(
            config_path=CONFIG_PATH,
            data_root=DATA_ROOT,
            results_root=RESULTS_ROOT,
            optimizers=[OPTIMIZER],
            seeds=SEEDS,
            device=DEVICE,
            resume=True,
            generate=False,
        )
    )

## Load the run-level data

`metrics.csv` contains normal train/validation trajectories.
`epoch_metrics.csv` contains the matched train/validation/test measurements at
nominal epochs 1–5. `spectral/summary.csv` contains WeightWatcher aggregates.

In [ ]:
metrics = load_metrics(
    RESULTS_ROOT,
    optimizers=[OPTIMIZER],
    seeds=SEEDS,
)
epoch_metrics = load_epoch_metrics(
    RESULTS_ROOT,
    optimizers=[OPTIMIZER],
    seeds=SEEDS,
)
spectral = load_spectral_metrics(
    RESULTS_ROOT,
    optimizers=[OPTIMIZER],
    seeds=SEEDS,
)
test_results = load_test_results(
    RESULTS_ROOT,
    optimizers=[OPTIMIZER],
    seeds=SEEDS,
)
test_summary = test_summary_table(test_results)

display(
    epoch_metrics[
        [
            "seed",
            "nominal_epoch",
            "step",
            "train_loss",
            "train_accuracy",
            "test_loss",
            "test_perplexity",
            "test_accuracy",
            "checkpoint_path",
        ]
    ]
)

## Publish the common baseline artifacts

This cell is part of the notebook contract. It writes the optimizer's raw and
summary CSVs under `baseline_reference/per_optimizer/<optimizer>/` and refreshes
the combined `all_runs/` and `summaries/` tables. The comparison notebook reads
those combined tables.

In [ ]:
exported_paths = export_optimizer_core_results(
    RESULTS_ROOT,
    BASELINE_STORE,
    optimizer=OPTIMIZER,
    seeds=SEEDS,
    sigma=BAND_SIGMA,
)
display(
    {
        name: str(path)
        for name, path in exported_paths.items()
    }
)

## Train and validation trajectories

In [ ]:
for metric in [
    "train_loss",
    "train_perplexity",
    "train_accuracy",
    "val_loss",
    "val_perplexity",
    "val_accuracy",
    "val_generalization_gap",
    "grad_norm_pre_clip",
    "update_to_weight_ratio",
    "tokens_per_sec",
]:
    plot_optimizer_band(
        metrics,
        optimizer=OPTIMIZER,
        metric=metric,
        sigma=BAND_SIGMA,
        title=f"{OPTIMIZER}: {metric} (mean ± 2 SD)",
    )
    plt.show()

## Matched train/test results by epoch

These are the preregistered epoch rows. The same nominal epoch is compared
across all three seeds.

In [ ]:
for metric in [
    "train_loss",
    "test_loss",
    "train_perplexity",
    "test_perplexity",
    "train_accuracy",
    "test_accuracy",
    "test_generalization_gap",
]:
    plot_optimizer_band(
        epoch_metrics,
        optimizer=OPTIMIZER,
        metric=metric,
        x="nominal_epoch",
        sigma=BAND_SIGMA,
        title=f"{OPTIMIZER}: {metric} by epoch (mean ± 2 SD)",
    )
    plt.show()

## WeightWatcher diagnostics

Only values returned by `watcher.analyze(ERG=True)` are plotted. Missing alpha
or ERG-gap values remain missing; no fallback is synthesized.

In [ ]:
for metric in [
    "alpha_median",
    "ERG_gap_median",
    "D_median",
    "stable_rank_median",
]:
    plot_optimizer_band(
        spectral,
        optimizer=OPTIMIZER,
        metric=metric,
        sigma=BAND_SIGMA,
        title=f"{OPTIMIZER}: {metric} (mean ± 2 SD)",
    )
    if metric == "alpha_median":
        plt.axhline(2.0, linestyle="--", linewidth=1.0)
    if metric == "ERG_gap_median":
        plt.axhline(0.0, linestyle="--", linewidth=1.0)
    plt.show()

## Final and validation-selected held-out test metrics

In [ ]:
display(
    test_summary[
        [
            "optimizer_label",
            "checkpoint",
            "metric",
            "n",
            "mean",
            "sd",
            "ci95_half_width",
            "ci95_lower",
            "ci95_upper",
        ]
    ]
)
for metric in [
    "test_loss",
    "test_perplexity",
    "test_accuracy",
]:
    plot_final_test_ci(
        test_summary,
        metric=metric,
        checkpoint="final",
        optimizers=[OPTIMIZER],
    )
    plt.show()

## Generate text from each final checkpoint

This qualitative diagnostic complements, but does not replace, held-out loss,
perplexity, and accuracy.

In [ ]:
RUN_GENERATION = True
sampling = CONFIG["sampling"]

if RUN_GENERATION:
    for seed in SEEDS:
        run_dir = run_directory(RESULTS_ROOT, OPTIMIZER, seed)
        checkpoint = run_dir / "checkpoint_final.pt"
        sample_seed = int(sampling["seed_offset"]) + int(seed)
        samples = generate_from_checkpoint(
            checkpoint,
            prompt=str(sampling["prompt"]),
            num_samples=int(sampling["num_samples"]),
            max_new_tokens=int(sampling["max_new_tokens"]),
            temperature=float(sampling["temperature"]),
            top_k=int(sampling["top_k"]),
            seed=sample_seed,
            device=DEVICE,
        )
        write_samples(
            run_dir,
            samples,
            prompt=str(sampling["prompt"]),
            checkpoint=checkpoint,
            settings={
                **sampling,
                "device": DEVICE,
                "seed": sample_seed,
            },
        )
        display(Markdown(f"### Seed {seed}"))
        for index, sample in enumerate(samples, 1):
            display(
                Markdown(
                    f"**Sample {index}**\n\n{sample}"
                )
            )